# 03 – Transformation Pipeline (Issue #7)

Notebook này minh hoạ cách sử dụng `src/cleaning_pipeline.py` – phần độc lập của Issue #7.

**Đây là dữ liệu thật.** Mọi ô dưới đây chạy trên
`data/interim/air_quality_canonical.parquet` và `data/interim/weather_canonical.parquet`:
9044 quan sát giờ, 2025-07-03 → 2026-07-15. Tên trạm và địa chỉ **không viết tay ở
đây** — ô 1 đọc trực tiếp từ dữ liệu và in ra, để không thể lệch với dữ liệu thật.

**Nhưng chưa có kết quả nghiên cứu nào.** Notebook chỉ dựng pipeline và đóng băng
dataset; chưa mô hình nào được huấn luyện, chưa có Recall, PR-AUC hay R². Các con
số in ra là **số liệu mô tả dữ liệu và số liệu kiểm chứng pipeline**, không phải kết
quả dự báo — không được trích dẫn như kết quả. Đánh giá mô hình thuộc Issue #11–#13.

Thứ tự bắt buộc là **merge → freeze → split → fit**:

- **merge** trước, để dùng đúng bản ghi.
- **freeze** sau merge và trước split, để metadata ghi lại phạm vi toàn bộ dataset.
- **split** trước **fit**, để mọi tham số học (median của `SimpleImputer`, IQR của
  `RobustScaler`) chỉ đến từ Train.

**Restart Kernel & Run All.** Các ô sau phụ thuộc kết quả ô trước.


In [1]:
import copy
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42  # Hạt giống cố định theo quy ước dự án
np.random.seed(RANDOM_STATE)

# Notebook chạy với cwd = notebooks/, nên thư mục gốc repo là "..".
# sys.path phải trỏ tới GỐC REPO để import được package `src.*` — trỏ vào
# `src/` rồi import `cleaning_pipeline` không tôn trọng cấu trúc package.
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.cleaning_pipeline import (  # noqa: E402
    add_cyclical_time_features,
    chronological_split,
    sort_by_time,
    build_preprocessing_pipeline,
    fit_pipeline_on_train,
    transform_with_pipeline,
    evaluate_log1p_transform,
    merge_air_weather,
    export_to_parquet,
    freeze_dataset,
    freeze_dataset_with_audit,
    run_preprocessing_audit,
    validate_no_leakage,
    NUMERIC_FEATURES,
    CYCLICAL_FEATURES,
    DIAGNOSTIC_FEATURES,
    TARGET_DERIVED_FLAGS,
    TARGET_CANDIDATE,
)


## 1. Dữ liệu thật từ `data/interim/`

Hai bảng canonical do Issue #3 sinh ra. Bảng khí tượng **không có cột
`station_id`** — Open-Meteo phục vụ theo toạ độ chứ không theo trạm — nên
`merge_air_weather()` sẽ tự rơi về `on=["timestamp"]`. Ô merge kiểm tra lại điều
này thay vì tin vào nó.

In [2]:
df_air = pd.read_parquet(REPO_ROOT / "data/interim/air_quality_canonical.parquet")
df_wx = pd.read_parquet(REPO_ROOT / "data/interim/weather_canonical.parquet")

print(f"Air quality: {df_air.shape}  ({df_air['timestamp'].min()} → {df_air['timestamp'].max()})")
print(f"Weather:     {df_wx.shape}  ({df_wx['timestamp'].min()} → {df_wx['timestamp'].max()})")
print(f"Air có station_id: {'station_id' in df_air.columns} | "
      f"Weather có station_id: {'station_id' in df_wx.columns}")

# Tên trạm / địa chỉ đọc thẳng từ dữ liệu — không viết tay trong markdown, vì
# bản trước đã ghi một tên trạm không tồn tại trong dữ liệu.
STATIONS = df_air["station_id"].unique().tolist()
print(f"\nStations ({len(STATIONS)}): {STATIONS}")
if "location" in df_air.columns:
    print("Locations:", df_air["location"].dropna().unique().tolist())

assert "station_id" not in df_wx.columns, (
    "Nếu weather_canonical có station_id thì merge phải dùng "
    "on=['station_id','timestamp'], không phải chỉ on=['timestamp']."
)
df_air.head(3)


Air quality: (9044, 8)  (2025-07-03 22:00:00+07:00 → 2026-07-15 17:00:00+07:00)
Weather:     (9072, 7)  (2025-07-03 00:00:00+07:00 → 2026-07-15 23:00:00+07:00)
Air có station_id: True | Weather có station_id: False

Stations (1): ['VN001_HANOI_556_NGUYEN_VAN_CU']
Locations: ['556 Nguyễn Văn Cừ']


,timestamp,station_id,location,pm25,pm10,pm25_was_stuck,pm25_was_missing,is_high_humidity_fog
0,2025-07-03 22:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,31.516667,64.686667,0,0,1
1,2025-07-03 23:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,33.672500,63.614167,0,0,1
2,2025-07-04 00:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,32.347000,56.636000,0,0,1


## 2. Merge với row-explosion protection

9.072 dòng khí tượng nhưng chỉ 9.044 dòng ô nhiễm: khí tượng phủ rộng hơn một
chút ở hai đầu chuỗi thời gian. Merge theo `timestamp` phải giữ **đúng** 9.044
dòng — thêm dòng nghĩa là join sai khoá.

`merge_air_weather()` tự kiểm tra ba điều và raise nếu vi phạm:

1. cột trùng tên ngoài khoá ghép (sẽ thành `_x`/`_y` và mất tên gốc),
2. khoá không unique (nguồn của row explosion),
3. **không cột khí tượng nào khớp được** — toàn bộ cột khí tượng là `NaN` nghĩa
   là lệch timezone/cửa sổ thời gian, và nếu im lặng thì `SimpleImputer` sẽ lấp
   median rồi mô hình vẫn "chạy mượt" trên 6 hằng số bịa.


In [3]:
df_merged = merge_air_weather(df_air, df_wx)
print(f"Merged: {df_merged.shape}  (air: {len(df_air)}, weather: {len(df_wx)})")

# Row explosion: phải bằng đúng, không phải "<=". Với `how="left"` + khoá
# unique hai phía + validate="1:1", "<=" là dead code — luôn đúng.
assert len(df_merged) == len(df_air), (
    f"Row Explosion: {len(df_merged)} != {len(df_air)}"
)

# Khí tượng phải thực sự khớp, không phải toàn NaN.
WX_COLS = [c for c in df_wx.columns if c != "timestamp"]
all_nan = [c for c in WX_COLS if df_merged[c].isna().all()]
assert not all_nan, f"Cột khí tượng toàn NaN: {all_nan}"
coverage = 100.0 * df_merged[WX_COLS].notna().all(axis=1).mean()
print(f'\u2713 No row explosion | coverage khí tượng: {coverage:.1f}%')

# `chronological_split()` TỪ CHỐI dữ liệu chưa sort thay vì tự sort lại — sort
# tại chỗ sẽ che giấu việc đường ống ingestion sinh ra thứ tự lộn xộn. Ở đây
# merge đã giữ thứ tự của bảng ô nhiễm; `sort_by_time()` là cách nói tường minh
# rằng thứ tự đó là đúng, thay vì để mặc định.
assert df_merged["timestamp"].is_monotonic_increasing, (
    "Merge đã làm mất thứ tự thời gian — phải sort tường minh bằng sort_by_time()"
)
df_merged = sort_by_time(df_merged)
print('\u2713 Sorted tường minh bằng sort_by_time()')
df_merged.head(3)


Merged: (9044, 14)  (air: 9044, weather: 9072)
✓ No row explosion | coverage khí tượng: 100.0%
✓ Sorted tường minh bằng sort_by_time()


,timestamp,station_id,location,pm25,pm10,pm25_was_stuck,pm25_was_missing,is_high_humidity_fog,temperature,relative_humidity,wind_speed,wind_direction,precipitation,surface_pressure
0,2025-07-03 22:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,31.516667,64.686667,0,0,1,26.1,95.0,0.36,304.0,0.0,1005.5
1,2025-07-03 23:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,33.672500,63.614167,0,0,1,25.7,98.0,0.85,320.0,0.0,1005.0
2,2025-07-04 00:00:00+07:00,VN001_HANOI_556_NGUYEN_VAN_CU,556 Nguyễn Văn Cừ,32.347000,56.636000,0,0,1,25.3,99.0,0.81,313.0,0.0,1004.9


## 3. Cyclical time features

Giờ trong ngày (0–23) và tháng trong năm (1–12) là biến tuần hoàn, không phải biến
tuyến tính. Mã hoá bằng sin/cos để 23:00 và 00:00 lân cận nhau thay vì ở hai đầu
trục. Không dùng raw hour/tháng vì `RobustScaler` sẽ coi 0 giờ và 23 giờ là hai
điểm cực đoan xa nhau.

In [4]:
df_merged = add_cyclical_time_features(df_merged)
print('Cyclical features:', CYCLICAL_FEATURES)
df_merged[['timestamp', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']].head(3)

Cyclical features: ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']


,timestamp,hour_sin,hour_cos,month_sin,month_cos
0,2025-07-03 22:00:00+07:00,-0.500000,0.866025,-0.5,-0.866025
1,2025-07-03 23:00:00+07:00,-0.258819,0.965926,-0.5,-0.866025
2,2025-07-04 00:00:00+07:00,0.000000,1.000000,-0.5,-0.866025


## 4. Dataset freeze (ghi nhận metadata)

Freeze **trước** split: metadata phải mô tả toàn bộ dataset đã đóng băng, không
phải riêng tập Train.

`freeze_dataset()` mặc định chỉ giữ **cột số** và loại:

- `timestamp`, `station_id` — không phải đặc trưng dự báo,
- target `pm25`,
- `TARGET_DERIVED_FLAGS` (`pm25_was_missing`) — hàm xác định của target, xem mục 6,
- cột không phải số (ví dụ `location` là chuỗi).

`station_count` là `None` khi dataset không có cột trạm, chứ không phải `0`:
"0 trạm" là khẳng định sai sự thật, còn `None` là "không đo được".


In [5]:
freeze_info = freeze_dataset(df_merged)
print(f'Rows:    {freeze_info.row_count}')
print(f'Span:    {freeze_info.timestamp_min} → {freeze_info.timestamp_max}')
print(f'Station: {freeze_info.station_count}')
print(f'Feature columns ({len(freeze_info.feature_columns)}): {freeze_info.feature_columns}')

Rows:    9044
Span:    2025-07-03 22:00:00+07:00 → 2026-07-15 17:00:00+07:00
Station: 1
Feature columns (13): ['pm10', 'pm25_was_stuck', 'is_high_humidity_fog', 'temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'precipitation', 'surface_pressure', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


## 5. Chronological split – vì sao `2026-01-15`

Split **tuyến tính theo thời gian**, tuyệt đối không random. Mốc cắt **không** chọn
theo tỷ lệ đẹp mắt (80/20, 70/30) mà theo **temporal representativeness +
event coverage** — hai tiêu chí mà `.agents/rules/analysis.md` đặt cho mọi phép
chia dữ liệu theo thời gian.

Lý do: dữ liệu chỉ có **một chu kỳ mùa** (07/2025–07/2026). Chia theo số dòng với
tập này gần như luôn dịch mốc cắt ra khỏi mùa đông. Bảng phân bố `pm25` theo tháng
ở ô dưới cho thấy đỉnh mùa đông rõ ở 2025-11 → 2026-01.

In [6]:
# --- 5a. Phân bố pm25 theo tháng --------------------------------------------
_df = df_merged.copy()
_df["month"] = _df["timestamp"].dt.strftime("%Y-%m")
_g = _df.groupby("month")["pm25"]
_idx = _g.size().index

monthly = pd.DataFrame({
    "n_gio": _g.size(),
    "n_co_pm25": _g.count(),
    "mean": _g.mean().round(1),
    "p90": _g.quantile(0.9).round(1),
    "max": _g.max().round(1),
    "gio>50": _df[_df["pm25"] > 50].groupby("month").size().reindex(_idx).fillna(0).astype(int),
    "gio>100": _df[_df["pm25"] > 100].groupby("month").size().reindex(_idx).fillna(0).astype(int),
})
display(monthly)

# 2026-07: trạm ngừng báo cáo từ 2026-07-01, chỉ còn 1 quan sát có giá trị.
# Hàng đó dựa trên n=1 nên KHÔNG dùng làm bằng chứng cho mốc cắt nào.
print(f"2026-07 chỉ có {int(monthly.loc['2026-07', 'n_co_pm25'])} quan sát có pm25 "
      f"trong {int(monthly.loc['2026-07', 'n_gio'])} giờ — hàng đó không đáng tin.")

# --- 5b. Các mốc cắt ứng viên ----------------------------------------------
# Cố tình dùng boolean mask thay vì `chronological_split()`: ở đây ta thử N
# mốc cắt để so sánh, và `chronological_split()` chỉ nhận một mốc mỗi lần
# gọi. Lựa chọn chính thức được thực hiện bằng chính hàm đó ở ô 13.
CANDIDATES = [
    ("2026-05-01", "80/20 — chia theo số dòng"),
    ("2026-03-24", "70/30 — chia theo số dòng"),
    ("2026-02-15", "60/40 — chia theo số dòng"),
    ("2026-01-15", "★ mùa đông / mùa đông — ĐÃ CHỌN"),
    ("2026-01-01", "đầu mùa đông"),
]

rows = []
for iso, note in CANDIDATES:
    cut = pd.Timestamp(f"{iso} 00:00", tz="Asia/Ho_Chi_Minh")
    tr = df_merged[df_merged["timestamp"] < cut]
    te = df_merged[df_merged["timestamp"] >= cut]
    tr_hi, te_hi = int((tr["pm25"] > 100).sum()), int((te["pm25"] > 100).sum())
    tr_p90, te_p90 = tr["pm25"].quantile(0.9), te["pm25"].quantile(0.9)
    rows.append({
        "cut": iso,
        "%train": round(100 * len(tr) / len(df_merged), 1),
        # Ba tiêu chí mà Issue #7 nêu: event coverage và tính đại diện theo mùa.
        "train_gio>100": tr_hi,
        "test_gio>100": te_hi,
        "%>100_trong_test": round(100 * te_hi / len(te), 2),
        # Tỷ lệ p90 test/train: càng gần 1.00 càng đại diện. Lệch xa 1 nghĩa là
        # Test không giống cái mà Train đã thấy — đó là phân phối lệch, không
        # phải kiểm chứng độc lập.
        "p90_test/train": round(te_p90 / tr_p90, 2),
        "p90_train": round(tr_p90, 1),
        "p90_test": round(te_p90, 1),
        "ghi_chu": note,
    })

display(pd.DataFrame(rows))


,n_gio,n_co_pm25,mean,p90,max,gio>50,gio>100
month,,,,,,,
2025-07,674,643,38.4,64.6,96.2,151,0
2025-08,744,683,29.6,51.2,79.2,78,0
2025-09,720,667,33.0,51.4,102.9,79,1
2025-10,744,698,37.9,65.8,96.9,194,0
2025-11,720,652,47.7,101.8,156.5,236,71
2025-12,744,623,75.5,119.6,148.2,464,141
2026-01,744,684,67.1,113.1,174.4,458,123
2026-02,672,651,47.2,82.0,198.9,202,33
2026-03,744,736,34.8,56.3,107.0,105,2


2026-07 chỉ có 1 quan sát có pm25 trong 354 giờ — hàng đó không đáng tin.


,cut,%train,train_gio>100,test_gio>100,%>100_trong_test,p90_test/train,p90_train,p90_test,ghi_chu
0,2026-05-01,79.9,388,6,0.33,0.66,86.0,56.7,80/20 — chia theo số dòng
1,2026-03-24,69.8,371,23,0.84,0.69,88.8,61.6,70/30 — chia theo số dòng
2,2026-02-15,60.0,364,30,0.83,0.66,91.8,60.2,60/40 — chia theo số dòng
3,2026-01-15,51.8,295,99,2.27,0.80,90.5,72.7,★ mùa đông / mùa đông — ĐÃ CHỌN
4,2026-01-01,48.1,213,181,3.85,0.97,85.1,82.3,đầu mùa đông


### Các tiêu chí mà Issue #7 yêu cầu, và số đo được cho từng mốc cắt

Issue #7 §Chronological Split đòi mốc cắt phải luận giải từ **ba** căn cứ khách
quan: event coverage, tính liên tục chuỗi, và **tính đại diện theo mùa**. Bảng ở ô
trên đo cả ba:

- `test_gio>100` / `%>100_trong_test` — **event coverage** của tập Test.
- `p90_test/train` — **tính đại diện theo mùa**: p90 của Test chia cho p90 của
  Train. Càng gần `1.00` càng giống.
- `train_gio>100` — Train có chứa regime ô nhiễm nặng hay không.

### 80/20, 70/30, 60/40 bị loại — không phải vì ít sự kiện trong Test

| mốc cắt | test_gio>100 | %>100 trong Test | p90 test/train |
|---|---|---|---|
| 2026-05-01 | 6 | 0,33% | 0,66 |
| 2026-03-24 | 23 | 0,84% | 0,69 |
| 2026-02-15 | 30 | 0,83% | 0,66 |

Cả ba đặt mốc **sau** đợt lạnh tháng 1, nên Test chỉ còn 6–30 giờ >100 — dưới 1%
trong mọi trường hợp. Với tỉ lệ nhãn nặng dưới 1%, mô hình dự báo "không cảnh báo"
cho mọi giờ vẫn đạt ~99% accuracy. Đó chính là **Accuracy Trap** mà
`.agents/rules/analysis.md` cấm, và làm Recall / PR-AUC ở Issue #11–#13 mất hết
ý nghĩa. `p90_test/train` ≈ 0,66–0,69 cho thấy đồng thời Test lệch hẳn về vùng
sạch hơn mà Train đã thấy.

### Đã chốt: `2026-01-15`. Không đổi sang `2026-01-01`.

**Lý do — tính đại diện theo thời gian.** Mốc `2026-01-15` cắt **giữa mùa đông**,
nên **cả Train lẫn Test đều chứa giai đoạn mùa đông**: Train có 295 giờ >100, Test
có 99. Điều này quyết định vì tập dữ liệu hiện chỉ có **đúng một chu kỳ mùa**
(07/2025–07/2026). Nếu Test chỉ chứa một mùa mà Train chưa từng trải qua, mọi
kết quả ở Issue #11–#13 sẽ đo trên phép dịch phân phối chứ không phải trên cùng
một thế giới khí quyển.

**Đây là lựa chọn phương pháp luận phù hợp với dữ liệu hiện có — KHÔNG phải tối ưu
tuyệt đối.** Nó không phải mốc cắt điểm cao nhất trên mọi tiêu chí, và không nên
được mô tả như vậy. `2026-01-01` có `test_gio>100` = 181 (cao hơn 99) và
`p90_test/train` = 0,97 (gần 1 hơn 0,80).

**Hạn chế đã biết — ghi lại cho Issues #11–#13:**

| Hạn chế | Số liệu |
|---|---|
| Mốc `2026-01-01` có event coverage cao hơn | 181 so với 99 |
| Mốc `2026-01-01` có p90 ratio gần 1 hơn | 0,97 so với 0,80 |
| Tỷ lệ nhãn nặng ở Test thấp hơn Train ~**2,8 lần** | Test 2,27% so với Train 6,30% |

Hệ quả cụ thể: **có dịch chuyển tỷ lệ lớp giữa Train và Test**. Khi đánh giá mô hình,
Recall và PR-AUC phải được diễn giải cẩn thận với sự dịch chuyển này; nếu cần có
thể báo cáo thêm chỉ số trên một tập con Test đã cân bằng nhãn. Việc chốt lại mốc
cắt **không thuộc phạm vi Issue #7** và chỉ nên xem xét khi có thêm một chu kỳ mùa nữa
trong dữ liệu.

### Vì sao giữ CẢ HAI phép kiểm tra số dòng

AC1 của Issue #7 chỉ định `assert len(df_merged) <= len(df_air)`. Phép này được
giữ nguyên để không lệch khỏi tiêu chí nghiệm thu. Nhưng với `how="left"` + khoá
unique hai phía + `validate="1:1"`, `<=` là **điều kiện cần mà không phân biệt được**
— luôn đúng, không bắt được lỗi nào. Vì vậy `merge_air_weather()` kiểm tra cả hai:

- `<=` — điều kiện cần: không được **sinh thêm** dòng (giữ nguyên theo AC1).
- `==` — bất biến thật của pipeline: không được **mất** dòng ô nhiễm nào.

Cơ sở của bất biến `==`: theo định nghĩa của Issue #7, phép ghép là
`merge(how="left")` theo khoá quan sát, nên **mọi dòng của bảng ô nhiễm đều được giữ
lại**. Giờ thiếu bản ghi khí tượng thì dòng ô nhiễm đó vẫn còn, chỉ mang `NaN` ở
các cột khí tượng — đó là hành vi `SimpleImputer` xử lý sau, không phải lý do để
xoá dòng. Pipeline này **không được phép loại bỏ air row trong bất kỳ trường hợp
hợp lệ nào**.

Nếu sau này pipeline được phép loại air row (ví dụ thêm bước loại quan sát không đạt
kiểm định chất lượng trước khi freeze), bất biến trở thành `<=` và phép `==` **phải
được nới lỏng** kèm ghi chú lý do trong mã. Đó là lý do hai phép cùng tồn tại thay
vì một phép bị xoá.

Đây là quyết định đã chốt. Bản ghi chính thức nằm ở `docs/roadmap.md`,
mục *Week 05 → Bản ghi chính thức — Các quyết định đã chốt cho Issue #7*.


In [7]:
# Mốc cắt cố định — xem lập luận ở mục 5. KHÔNG suy ra từ quantile của frame.
SPLIT_TIMESTAMP = pd.Timestamp("2026-01-15 00:00", tz="Asia/Ho_Chi_Minh")

train_df, test_df = chronological_split(df_merged, SPLIT_TIMESTAMP)

print(f"Split timestamp: {SPLIT_TIMESTAMP}")
print(f"Train: {len(train_df)} dòng ({100 * len(train_df) / len(df_merged):.1f}%)")
print(f"Test:  {len(test_df)} dòng ({100 * len(test_df) / len(df_merged):.1f}%)")
print(f"Train: {int((train_df['pm25'] > 50).sum())} giờ >50, {int((train_df['pm25'] > 100).sum())} giờ >100")
print(f"Test:  {int((test_df['pm25'] > 50).sum())} giờ >50, {int((test_df['pm25'] > 100).sum())} giờ >100")

# Validation cứng: không quan sát nào của Test được phép nằm trước mốc Train.
assert train_df["timestamp"].max() < test_df["timestamp"].min()
print("✓ No temporal leakage")

Split timestamp: 2026-01-15 00:00:00+07:00
Train: 4682 dòng (51.8%)
Test:  4362 dòng (48.2%)
Train: 1425 giờ >50, 295 giờ >100
Test:  878 giờ >50, 99 giờ >100
✓ No temporal leakage


### 5c. Ba cột cờ chẩn đoán — nhưng không phải cờ nào cũng làm feature

Ba cột cờ do Issue #5 sinh ra. Chúng là **cờ chẩn đoán, KHÔNG phải giá trị đã được
thay thế**: Issue #6 đã khẳng định qua `assert_no_imputation()` rằng quan sát được
chỉ có thể biến mất, không bao giờ bị bịa giá trị.

| cột | nghĩa | làm feature? |
|---|---|---|
| `is_high_humidity_fog` | sương mù quang học — cảm biến đọc sai, không phải ô nhiễm thật | ✅ có |
| `pm25_was_stuck` | cảm biến kẹt, chuỗi 0.0 dài | ✅ có (xem dưới) |
| `pm25_was_missing` | trạm ngừng báo cáo — thiếu hẳn quan sát | ❌ **không** |

**Vì sao `pm25_was_missing` không được làm feature.** Nó bằng đúng `pm25.isna()`.
Mà `SimpleImputer(strategy="median")` lại điền median cho **đúng những hàng đó**.
Hệ quả: mọi hàng `flag == 1` có target bằng đúng median của tập học — đo trên dữ
liệu thật là **100%**, ở cả Train lẫn Test. Mô hình học được quy tắc
`flag == 1 ⇒ pm25 == median` và đúng 100%: đó là rò rỉ target theo **cấu trúc**.

Điểm mấu chốt: `validate_no_leakage()` **không** bắt được lỗi này, vì imputer và
scaler vẫn học đúng trên Train. Guard kiểm tham số học thì vô dụng ở đây — phải loại
ở mức danh sách feature. Ô code dưới đây đo lại trên dữ liệu thật thay vì chỉ
khẳng định bằng lời.

**`pm25_was_stuck` là hằng số (toàn 0) trên tập này.** Đây là **phát hiện thực
nghiệm, không phải lỗi**: Issue #5 đã kết luận chuỗi 0.0 dài ở Hà Nội là hiện
tượng tự nhiên chứ không phải cảm biến hỏng. Vì vậy cột này **không mang thông tin
phân biệt** và sẽ không giúp mô hình dự đoán gì.

Giữ lại cột vẫn an toàn về kỹ thuật: `RobustScaler` với cột hằng cho `scale_ = 1.0`,
`center_ = 0.0`, output `0.0` — không NaN, không crash. Giữ nó để Issue #5 và các
notebook sau dùng chung một bộ cột, thay vì phải giải thích vì sao một cột biến mất.


In [8]:
for _c in TARGET_DERIVED_FLAGS + DIAGNOSTIC_FEATURES:
    if _c in df_merged.columns:
        print(f"{_c:24s} nunique={df_merged[_c].nunique()}  sum={int(df_merged[_c].sum())}")

# `nunique() == 1` chỉ nói "hằng số", không nói hằng số *nào*. Markdown khẳng
# định "toàn 0" nên phải kiểm đúng con số đó.
assert int(df_merged["pm25_was_stuck"].sum()) == 0, (
    "pm25_was_stuck không còn toàn 0 — mốc chẩn đoán của Issue #5 cần được cập nhật."
)
print("\n✓ Xác nhận: pm25_was_stuck hằng số 0 → không mang thông tin phân biệt trên tập này.")

# --- Đo lại cơ chế rò rỉ của pm25_was_missing trên dữ liệu thật --------------
# Không chỉ khẳng định bằng lời: chạy đúng imputer mà pipeline sẽ dùng, rồi đếm
# xem bao nhiêu hàng `flag == 1` có target bằng median Train sau khi điền.
_flag = "pm25_was_missing"
# Imputer phải học trên đúng tập cột mà pipeline sẽ nhận (numeric + cờ được
# dùng), KHÔNG chỉ `NUMERIC_FEATURES` — nếu không median sẽ lệch với median thật.
_impute_cols = [c for c in NUMERIC_FEATURES if c in df_merged.columns] + [
    c for c in DIAGNOSTIC_FEATURES if c in df_merged.columns
]
_impute_frame = df_merged[_impute_cols + [TARGET_CANDIDATE]]

_imputer = SimpleImputer(strategy="median").fit(
    train_df[_impute_cols + [TARGET_CANDIDATE]]
)
_median = float(_imputer.statistics_[-1])  # cột target nằm cuối
_filled = _imputer.transform(test_df[_impute_cols + [TARGET_CANDIDATE]])[:, -1]
_flagged = test_df[_flag].to_numpy().astype(bool)

# Tính lại median từ Train bằng pandas để không tin tuyệt đối vào thứ tự cột.
_median_pandas = float(train_df[TARGET_CANDIDATE].median())
assert np.isclose(_median, _median_pandas), (
    f"median từ SimpleImputer ({_median:.4f}) lệch pandas ({_median_pandas:.4f})"
)
print(f"\nMedian {TARGET_CANDIDATE} học từ Train: {_median:.4f} (khớp pandas.median)")
print(f"Hàng Test có {_flag}=1: {int(_flagged.sum())}")
if _flagged.sum():
    n_match = int(np.isclose(_filled[_flagged], _median).sum())
    print(f"Trong đó có target == median sau khi impute: {n_match}/{int(_flagged.sum())} "
          f"({100 * n_match / _flagged.sum():.1f}%)")
    assert n_match == int(_flagged.sum()), (
        "Tỷ lệ khớp không phải 100% — lập luận loại cờ này cần được viết lại."
    )
print("✓ Cơ chế rò rỉ được đo và xác nhận trên dữ liệu thật, không phải khẳng định suông.")


pm25_was_missing         nunique=2  sum=1289
pm25_was_stuck           nunique=1  sum=0
is_high_humidity_fog     nunique=2  sum=2834

✓ Xác nhận: pm25_was_stuck hằng số 0 → không mang thông tin phân biệt trên tập này.

Median pm25 học từ Train: 37.8300 (khớp pandas.median)
Hàng Test có pm25_was_missing=1: 1033
Trong đó có target == median sau khi impute: 1033/1033 (100.0%)
✓ Cơ chế rò rỉ được đo và xác nhận trên dữ liệu thật, không phải khẳng định suông.


## 6. Build & fit Pipeline (CHỈ trên Train)

**Về target.** `NUMERIC_FEATURES` của module **không** chứa `pm25` nữa, và
`build_preprocessing_pipeline()` raise `ValueError` nếu target lọt vào danh sách.
Bản gốc liệt kê sẵn `pm25`, nên gọi hàm không tham số là đưa chính câu trả lời vào
`X`: bài toán dự báo biến thành bài toán đọc lại đáp án, và `validate_no_leakage()`
**không** bắt được lỗi đó vì imputer vẫn học đúng trên Train.

**Về cờ chẩn đoán.** Ba cờ của Issue #6 không cùng một loại:

| Cờ | Dùng làm feature? | Lý do |
|---|---|---|
| `is_high_humidity_fog` | ✅ có | sương mù quang học — điều kiện khí quyển, hợp lệ dự báo |
| `pm25_was_stuck` | ✅ có | cảm biến kẹt (trên tập Hà Nội hiện là hằng số 0, vô hại) |
| `pm25_was_missing` | ❌ **không** | bằng `pm25.isna()`, mà `SimpleImputer` lại điền median cho đúng những hàng đó |

Với `pm25_was_missing`, mọi hàng `flag == 1` có target bằng **đúng** median của
tập học. Ô 15 **đo lại** điều này trên dữ liệu thật: 256/256 hàng ở Train và
1.033/1.033 hàng ở Test, tức **100% ở cả hai**. Mô hình học được quy tắc
`flag == 1 ⇒ pm25 == median` và đúng 100%: đó là rò rỉ target theo **cấu trúc**,
và không guard nào kiểm tham số học có bắt được — vì imputer vẫn học đúng trên
Train. Phải loại ở mức danh sách feature. Cờ vẫn được giữ trong dataset như tài
liệu chẩn đoán.

Đọc con số: tổng số hàng `pm25` bị `NaN` là **1.549** — **1.289** mang cờ này
(khối dài) và **260** là khối ngắn không mang cờ. Dòng rò rỉ đo được là 256 hàng
ở Train, không phải 1.289, vì cờ chỉ bắt khối dài.

Hai nhóm cờ được dùng đều là 0/1 nên đi chung nhánh `numeric` (impute median +
robust scale) cùng các đặc trưng số khác.


In [9]:
# Cờ chẩn đoán ĐƯỢC dùng làm feature — lấy từ hằng của module, không liệt kê
# tay ở đây, để không trôi khỏi quyết định của Issue #7.
flag_feats = [c for c in DIAGNOSTIC_FEATURES if c in df_merged.columns]

numeric_feats = [c for c in NUMERIC_FEATURES if c in df_merged.columns]
cyc_feats = [c for c in CYCLICAL_FEATURES if c in df_merged.columns]

feature_cols = numeric_feats + cyc_feats + flag_feats
print(f"Numeric:     {numeric_feats}")
print(f"Cyclical:    {cyc_feats}")
print(f"Flags:       {flag_feats}")
print(f"Target:      {TARGET_CANDIDATE}  (KHÔNG đưa vào feature)")

# Ba cái bảo vệ lặp lại, vì mỗi cái bắt một loại lỗi:
#   1. target lọt vào feature
assert TARGET_CANDIDATE not in feature_cols, "Target rò rỉ vào feature!"
#   2. cờ là hàm xác định của target (rò rỉ cấu trúc)
for flag in TARGET_DERIVED_FLAGS:
    assert flag not in feature_cols, (
        f"{flag} là hàm xác định của {TARGET_CANDIDATE} — đưa vào feature là "
        f"rò rỉ mà validate_no_leakage() không bắt được."
    )
#   3. khoá định danh không phải đặc trưng dự báo
assert "timestamp" not in feature_cols, "timestamp không phải feature số!"
assert "station_id" not in feature_cols, "station_id không phải feature số!"

# Và module phải tự chặn target nếu ai đó truyền sai.
try:
    build_preprocessing_pipeline(numeric_features=["pm25"], cyclical_features=[])
    raise SystemExit("build_preprocessing_pipeline() chưa chặn target!")
except ValueError as exc:
    print(f"\n✓ Module tự chặn target: {exc}")

pipeline = build_preprocessing_pipeline(
    numeric_features=numeric_feats + flag_feats,
    cyclical_features=cyc_feats,
)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

# Fit STRICTLY trên Train
pipeline = fit_pipeline_on_train(pipeline, X_train)
print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")
print("✓ Pipeline fitted on Train only")


Numeric:     ['pm10', 'temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'precipitation', 'surface_pressure']
Cyclical:    ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']
Flags:       ['pm25_was_stuck', 'is_high_humidity_fog']
Target:      pm25  (KHÔNG đưa vào feature)

✓ Module tự chặn target: Target 'pm25' không được đưa vào feature. Đưa target vào X biến bài toán dự báo thành bài toán đọc lại câu trả lời, khiến mọi chỉ số ở Issue #11-#13 mất ý nghĩa.

X_train: (4682, 13) | X_test: (4362, 13)
✓ Pipeline fitted on Train only


## 7. Transform Train & Test

Cùng một pipeline đã fit trên Train áp cho cả hai tập. `transform_with_pipeline()`
chỉ gọi `.transform()`, không bao giờ `.fit()`, nên median và IQR đã học từ Train là
thứ duy nhất dùng cho Test.

In [10]:
X_train_transformed = transform_with_pipeline(pipeline, X_train)
X_test_transformed = transform_with_pipeline(pipeline, X_test)

print(f'X_train_transformed: {X_train_transformed.shape}')
print(f'X_test_transformed: {X_test_transformed.shape}')
print(f'NaN sau transform:  train={np.isnan(X_train_transformed).sum()}, '
      f'test={np.isnan(X_test_transformed).sum()}')
print('✓ Test transformed using Train-fitted parameters (no leakage)')

X_train_transformed: (4682, 13)
X_test_transformed: (4362, 13)
NaN sau transform:  train=0, test=0
✓ Test transformed using Train-fitted parameters (no leakage)



## 7b. Quan hệ với các hàng `pm25_was_missing`

Issue #6 bàn giao rõ một ràng buộc: **1.289 hàng mang cờ `pm25_was_missing = 1`
(khối khuyết > 6 giờ) — Issue #7 tuyệt đối không được nội suy mù các hàng này.**
(Tổng số hàng `pm25` bị `NaN` là 1.549; 260 hàng còn lại là khối ngắn, không mang
cờ, nhưng cũng không bị điền trong dataset.)

Cách pipeline này tuân thủ ràng buộc đó, và đây là điểm cần hiểu rõ:

- **Không ghi đè bất kỳ giá trị nào trong dataset.** `df_merged` và file xuất ra giữ
  nguyên `NaN`. Ô `assert_no_imputation()` của Issue #6 vẫn PASS sau khi chạy
  notebook này. Việc điền median **không diễn ra trong dữ liệu**.
- Việc điền chỉ xảy ra **bên trong `ColumnTransformer`**, lúc `transform()`, bằng
  median học từ **Train**. Đây là bước chuẩn bị feature cho mô hình, không phải sửa
  dữ liệu — và tham số median đến từ Train nên không rò rỉ.
- Cột cờ `pm25_was_missing` **không** được đưa vào feature — vì như mục 6 đã
  chứng minh, đưa nó vào là rò rỉ target theo cấu trúc. Hai cờ còn lại
  (`pm25_was_stuck`, `is_high_humidity_fog`) thì có.

Điểm cần lưu ý cho Issue #11–#13: khi đánh giá mô hình, `pm25` vẫn là `NaN` ở các
hàng này, nên tập đánh giá phải quyết định rõ xử lý `NaN` ở **target** — đây là việc
của issue phân loại, không phải việc của pipeline tiền xử lý này.


In [11]:

# Bao nhiêu ô thực sự được điền median, và tham số học từ đâu?
# "num" là Pipeline([imputer, scaler]) nên tham số nằm ở named_steps bên trong nó,
# không phải .statistics_ trực tiếp.
_num_pipe = pipeline.named_steps["preprocessor"].named_transformers_["num"]
_imputer = _num_pipe.named_steps["imputer"]
_scaler = _num_pipe.named_steps["scaler"]

_cols = numeric_feats + flag_feats
print("Median (SimpleImputer) và scale (RobustScaler) — đều học trên Train:")
for _f, _med, _cen, _scl in zip(
    _cols, _imputer.statistics_, _scaler.center_, _scaler.scale_
):
    print(f"  {_f:24s} median={_med:>9.4f}  center={_cen:>9.4f}  scale={_scl:.4f}")

# pm25 không nằm trong feature (là target) nên không bị impute ở đây.
assert TARGET_CANDIDATE not in numeric_feats + flag_feats

# Dataset bản gốc vẫn còn NaN — không có giá trị nào bị ghi đè.
_nan_pm25 = int(df_merged["pm25"].isna().sum())
_nan_pm10 = int(df_merged["pm10"].isna().sum())
_flagged = int(df_merged["pm25_was_missing"].sum())
print(f"\nNaN còn lại trong df_merged: pm25={_nan_pm25}, pm10={_nan_pm10}")
print(f"Số hàng mang cờ pm25_was_missing: {_flagged}")
assert _nan_pm25 > 0, "Dataset đã bị ghi đè giá trị — vi phạm ràng buộc của Issue #6!"
print("✓ Dataset không bị ghi đè: giá trị thiếu vẫn là NaN, chỉ được impute ở transform().")

Median (SimpleImputer) và scale (RobustScaler) — đều học trên Train:
  pm10                     median=  54.3282  center=  54.3282  scale=42.5583
  temperature              median=  25.2000  center=  25.2000  scale=7.1750
  relative_humidity        median=  83.0000  center=  83.0000  scale=22.0000
  wind_speed               median=   2.0100  center=   2.0100  scale=1.6400
  wind_direction           median= 133.0000  center= 133.0000  scale=171.0000
  precipitation            median=   0.0000  center=   0.0000  scale=0.1000
  surface_pressure         median=1009.1000  center=1009.1000  scale=12.2000
  pm25_was_stuck           median=   0.0000  center=   0.0000  scale=1.0000
  is_high_humidity_fog     median=   0.0000  center=   0.0000  scale=1.0000

NaN còn lại trong df_merged: pm25=1549, pm10=1469
Số hàng mang cờ pm25_was_missing: 1289
✓ Dataset không bị ghi đè: giá trị thiếu vẫn là NaN, chỉ được impute ở transform().


## 8. log1p — chẩn đoán trên Train, KHÔNG áp dụng

Đo skew và kurtosis của `pm25` trên **Train** trước và sau `log1p`, chỉ để có số
liệu khi quyết định ở issue sau.

**Quyết định đã chốt: KHÔNG transform target.** Giữ `pm25` ở µg/m³ để ngưỡng cảnh
báo còn nghĩa trực tiếp với bối cảnh chất lượng không khí, và phần downstream dễ
đọc hơn. AC #7 chỉ yêu cầu "khảo sát", không yêu cầu áp dụng.

In [12]:
y_train = train_df[TARGET_CANDIDATE]
log1p_diag = evaluate_log1p_transform(y_train)

print(f"skew:     {log1p_diag['original_skew']:.3f}  →  log1p: {log1p_diag['log1p_skew']:.3f}")
print(f"kurtosis: {log1p_diag['original_kurtosis']:.3f}  →  log1p: {log1p_diag['log1p_kurtosis']:.3f}")
print(f"n = {int(y_train.notna().sum())} quan sát Train có pm25")
print("\n→ Chỉ là chẩn đoán. Target giữ nguyên ở µg/m³, KHÔNG áp dụng log1p.")

skew:     1.148  →  log1p: -0.656
kurtosis: 1.068  →  log1p: 0.726
n = 4279 quan sát Train có pm25

→ Chỉ là chẩn đoán. Target giữ nguyên ở µg/m³, KHÔNG áp dụng log1p.


## 9. Tái sử dụng module audit của Issue #5

`run_preprocessing_audit()` gọi lại `audit_dataframe()` + `audit_six_dimensions()`
từ `src/data_quality.py`, tức là đoạn audit chạy trên canonical interim ở notebook
02 được dùng lại nguyên vẹn thay vì viết lại bộ quy tắc.

In [13]:
audit_results = run_preprocessing_audit(df_merged, dataset_name='air_pollution_final')

print('Summary table:')
display(audit_results['summary_table'])

print('\nSix dimensions:', list(audit_results['six_dimensions']['dimensions'].keys()))

Summary table:


,column_name,dtype,total_count,missing_count,missing_percentage,unique_count,min,max,q25,q50,q75
0,timestamp,"datetime64[ns, Asia/Ho_Chi_Minh]",9044,0,0.0000,9044,2025-07-03 22:00:00+07:00,2026-07-15 17:00:00+07:00,NaN,NaN,NaN
1,station_id,object,9044,0,0.0000,1,VN001_HANOI_556_NGUYEN_VAN_CU,VN001_HANOI_556_NGUYEN_VAN_CU,NaN,NaN,NaN
2,location,object,9044,0,0.0000,1,556 Nguyễn Văn Cừ,556 Nguyễn Văn Cừ,NaN,NaN,NaN
3,pm25,float64,9044,1549,17.1274,7346,1.06,198.946667,24.992917,3.731583e+01,55.725455
4,pm10,float64,9044,1469,16.2428,7437,1.01,339.994286,37.883442,5.516545e+01,81.642639
5,pm25_was_stuck,int64,9044,0,0.0000,1,0.0,0.0,0.000000,0.000000e+00,0.000000
6,pm25_was_missing,int64,9044,0,0.0000,2,0.0,1.0,0.000000,0.000000e+00,0.000000
7,is_high_humidity_fog,int64,9044,0,0.0000,2,0.0,1.0,0.000000,0.000000e+00,1.000000
8,temperature,float64,9044,0,0.0000,283,8.9,38.6,21.300000,2.540000e+01,28.400000
9,relative_humidity,float64,9044,0,0.0000,68,30.0,100.0,71.000000,8.300000e+01,92.000000



Six dimensions: ['completeness', 'accuracy', 'consistency', 'validity', 'uniqueness', 'timeliness']


## 10. Leakage validation

`validate_no_leakage()` kiểm **bốn** lớp, mỗi lớp vá một lỗ hổng của bản gốc:

1. **Thứ tự thời gian** — `train.max() < test.min()`. Bản gốc không đọc cột
   `timestamp` lần nào, nên một `train_test_split` ngẫu nhiên vẫn in ra
   "✓ No leakage".
2. **Giá trị học được, không chỉ sự tồn tại** — so **mọi** tham số đã học
   (`statistics_`, `center_`, `scale_`, …) với giá trị refit trên Train. Bản gốc
   kiểm `hasattr()` rồi bỏ qua, nên chỉ so `imputer.statistics_`; thay
   `RobustScaler` bằng bản fit trên Test vẫn qua.
3. **Mọi nhánh của `ColumnTransformer`** — hàm đi theo cây `named_steps` nên tự
   bao phủ nhánh thêm sau này. Bản gốc hard-code tên `"num"`.
4. **Sức phân biệt của phép so sánh** — từ chối chứng nhận nếu refit trên Train và
   trên Test cho ra cùng tham số, vì khi đó việc "khớp Train" không chứng minh
   được điều gì.

`X_train`/`X_test` là **keyword-only**: truyền sai thứ tự là lỗi ghi nhầm biến, và
Python chặn ngay thay vì để guard phải đoán.

Chi tiết: `tests/test_cleaning_pipeline_guards.py`.


In [14]:
# timestamps/test_timestamps là tùy chọn: truyền vào thì lớp kiểm chứng
# thứ tự thời gian được bật. Ở đây ta BẬT nó — nếu split bị đảo sang ngẫu nhiên
# thì ô này phải ném lỗi, không phải in ra dấu tick.
validate_no_leakage(
    pipeline,
    X_train=X_train,
    X_test=X_test,
    timestamps=train_df["timestamp"],
    test_timestamps=test_df["timestamp"],
)
print('✓ Pipeline validated: fitted on Train only, no leakage')

# Và phải bắt được biến thể rò rỉ: thay scaler bằng bản fit trên Test.
# `deepcopy` chứ không `clone`: `clone` trả về bản CHƯA fit, nên không có
# `transformers_` để sửa — và "sửa scaler của bản chưa fit" là vô nghĩa.
# Fit đúng tập cột của nhánh `num` — scaler chỉ thấy cột numeric.
_num_cols = numeric_feats + flag_feats
_leaky = copy.deepcopy(pipeline)
_leaky.named_steps["preprocessor"].named_transformers_["num"] \
    .named_steps["scaler"].fit(X_test[_num_cols])
try:
    validate_no_leakage(_leaky, X_train=X_train, X_test=X_test)
    raise SystemExit("Guard KHÔNG bắt được scaler fit trên Test!")
except AssertionError as exc:
    print(f"\n✓ Guard bắt được scaler rò rỉ: {str(exc).splitlines()[0]}")


✓ Pipeline validated: fitted on Train only, no leakage

✓ Guard bắt được scaler rò rỉ: Temporal/preprocessing leakage — tham số đã học KHÔNG khớp giá trị refit trên Train:


## 11. Freeze dataset with audit

Đóng băng kèm kết quả audit của Issue #5, để metadata của dataset mang theo bằng
chứng chất lượng chứ không chỉ số dòng.

In [15]:
freeze_info, audit = freeze_dataset_with_audit(
    df_merged,
    feature_columns=feature_cols,
    additional_info={"split_timestamp": str(SPLIT_TIMESTAMP)},
    run_audit=True,
)

print(f'Rows:    {freeze_info.row_count}')
print(f'Span:    {freeze_info.timestamp_min} → {freeze_info.timestamp_max}')
print(f'Station: {freeze_info.station_count}')
print(f'Split timestamp: {freeze_info.additional_info["split_timestamp"]}')
print(f'Feature columns ({len(freeze_info.feature_columns)}): {freeze_info.feature_columns}')
print(f'Audit recorded:   {"audit" in freeze_info.additional_info}')

Rows:    9044
Span:    2025-07-03 22:00:00+07:00 → 2026-07-15 17:00:00+07:00
Station: 1
Split timestamp: 2026-01-15 00:00:00+07:00
Feature columns (13): ['pm10', 'temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'precipitation', 'surface_pressure', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'pm25_was_stuck', 'is_high_humidity_fog']
Audit recorded:   True


## 12. Xuất `data/processed/air_pollution_final.parquet`

`export_to_parquet()` ghi **nguyên tử**: ghi file tạm cùng thư mục, validate
(số dòng, tên cột, **dtype**) rồi mới `os.replace()` sang đích. Bản gốc ghi thẳng
vào đích nên validation fail để lại một artifact hỏng ngay tại đường dẫn chính.

Hàm cũng **từ chối ghi đè** file đã tồn tại trừ khi `overwrite=True`. Dưới đây là
`True` vì notebook chạy lại nhiều lần trên cùng một artifact.


In [16]:
OUTPUT_PATH = REPO_ROOT / "data/processed/air_pollution_final.parquet"

# overwrite=True: notebook chạy lại nhiều lần trên cùng artifact. Nếu không có
# cờ này thì lần chạy thứ hai sẽ dừng ở FileExistsError.
export_to_parquet(df_merged, OUTPUT_PATH, overwrite=True)

# Đọc lại độc lập: export_to_parquet() đã tự validate, nhưng kiểm tra ở đây
# in ra con số để đối chiếu với bảng ở trên.
_readback = pd.read_parquet(OUTPUT_PATH)
print(f'✓ Exported: {OUTPUT_PATH.relative_to(REPO_ROOT)}')
print(f'  rows:   {len(_readback)} (expected {len(df_merged)})')
print(f'  cols:   {len(_readback.columns)}')
print(f'  size:   {OUTPUT_PATH.stat().st_size / 1024:.0f} KB')
print(f'  dtypes: {sorted({str(t) for t in _readback.dtypes})}')
assert len(_readback) == len(df_merged)
assert list(_readback.columns) == list(df_merged.columns)

# Không còn file tạm sót lại sau một lần export thành công.
assert not list(OUTPUT_PATH.parent.glob("*.tmp")), "file tạm sót lại"


✓ Exported: data\processed\air_pollution_final.parquet
  rows:   9044 (expected 9044)
  cols:   18
  size:   261 KB
  dtypes: ['datetime64[ns, Asia/Ho_Chi_Minh]', 'float64', 'int64', 'object']


## Tổng kết

| Bước | Trạng thái |
|---|---|
| Đọc dữ liệu thật từ `data/interim/` | ✓ |
| Merge + row explosion protection | ✓ |
| Cyclical time features | ✓ |
| Dataset freeze helper | ✓ |
| Chronological split tại `2026-01-15` | ✓ |
| Target loại khỏi feature | ✓ |
| 2/3 cờ chẩn đoán vào feature (`pm25_was_missing` bị loại — rò rỉ target) | ✓ |
| Pipeline fit trên Train only | ✓ |
| Transform Train/Test | ✓ |
| log1p diagnostic (không áp dụng) | ✓ |
| Tái sử dụng audit của Issue #5 | ✓ |
| Leakage validation | ✓ |
| Freeze with audit | ✓ |
| NaN được điền median **trong pipeline**, không ghi đè dataset | ✓ |
| Xuất `data/processed/air_pollution_final.parquet` | ✓ |

**Còn lại cho các issue sau:** chưa huấn luyện mô hình nào, chưa có Recall, PR-AUC
hay R². Việc đó thuộc Issue #11–#13. Notebook này chỉ chứng minh pipeline chạy
đúng trên dữ liệu thật và không rò rỉ.

**Một việc phải quyết ở issue sau:** `pm25` vẫn là `NaN` ở **1.549 hàng**
(1.289 khối dài có cờ, 260 khối ngắn). Khi đánh giá mô hình phải chốt cách xử lý
`NaN` ở **target** — đây không phải việc của pipeline tiền xử lý, và không được
âm thầm lấp bằng giá trị nào.

**Mốc cắt — đã chốt `2026-01-15`.** Lý do: mốc này cắt giữa mùa đông nên cả
Train lẫn Test đều chứa giai đoạn mùa đông, điều quyết định vì tập dữ liệu chỉ
có **một** chu kỳ mùa. Đây là lựa chọn phương pháp luận phù hợp với dữ liệu hiện
có, **không phải tối ưu tuyệt đối**: hạn chế đã biết là tỷ lệ nhãn nặng ở Test
(2,27%) thấp hơn Train (6,30%) ~2,8 lần. Chi tiết ở mục 5 và `docs/roadmap.md`.